## Libraries & Organizing

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import jinja2

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_BATCHES =  PROJECT_ROOT / "data" / "batches"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

In [ ]:
final = pd.read_csv(DATA_PROCESSED / "final_dataset_NAs.csv")


In [ ]:
final.columns

In [ ]:
desc = final[
    [
        "forward_looking_intensity",
        "dict_score",
        "specificity",
        "economic_substance",
        "tone",
        "certainty",
        "car_immediate",
        "car_medium",
        "car_long",
        "dict_score",
        "bm",
        "log_assets",
        "log_word_count"
    ]
].describe().T

# Rename rows for LaTeX
desc.index = [
    "forward\\_looking\\_intensity",
    "dict_score",
    "specificity",
    "economic\\_substance",
    "tone",
    "certainty",
    "car\\_immediate",
    "car\\_medium",
    "car\\_long",
    "dict\\_score",
    "bm",
    "log\\_assets",
    "log\\_word_count"
]

print(
    desc.to_latex(
        float_format="%.3f",
        caption="Descriptive statistics",
        label="tab:llama_descriptive_stats",
        escape=False
    )
)

In [ ]:
import pandas as pd

qual_cols = [
    "main_focus",
    "secondary_focus",
    "managerial_horizon",
    "overall_outlook"
]

def make_pct_table(df, col, top_n=None):
    tab = (
        df[col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
        .reset_index()
    )
    tab.columns = ["Category", "Percentage"]

    if top_n is not None and len(tab) > top_n:
        top = tab.iloc[:top_n].copy()
        other_pct = tab.iloc[top_n:]["Percentage"].sum().round(2)
        tab = pd.concat(
            [top, pd.DataFrame([{"Category": "Other", "Percentage": other_pct}])],
            ignore_index=True
        )

    return tab

# Example: compact appendix tables
main_focus_tab = make_pct_table(final, "main_focus", top_n=15)
secondary_focus_tab = make_pct_table(final, "secondary_focus", top_n=15)
horizon_tab = make_pct_table(final, "managerial_horizon")
outlook_tab = make_pct_table(final, "overall_outlook")

In [ ]:
print(
    main_focus_tab.to_latex(
        index=False,
        caption="Distribution of Main Focus Categories",
        label="tab:main_focus_distribution",
        float_format="%.2f"
    )
)

In [ ]:
print(
    secondary_focus_tab.to_latex(
        index=False,
        caption="Distribution of Secondary Focus Categories",
        label="tab:secondary_focus_distribution",
        float_format="%.2f"
    )
)


In [ ]:
print(
    horizon_tab.to_latex(
        index=False,
        caption="Distribution of Managerial Horizon Categories",
        label="tab:managerial_horizon_distribution",
        float_format="%.2f"
    )
)


In [ ]:

print(
    outlook_tab.to_latex(
        index=False,
        caption="Distribution of Overall Outlook Categories",
        label="tab:overall_outlook_distribution",
        float_format="%.2f"
    )
)